In [4]:
import shap
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder

# Load trained classifier and source data
df = pd.read_csv("../data/processed/telco_cleaned.csv")
model = joblib.load("../artifacts/best_classifier.pkl")

# Recreate the same feature preprocessing used for training
y = df["churn"].map({"No": 0, "Yes": 1})
X = df.drop(columns=["customerid", "churn"], errors="ignore").copy()

for col in X.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])

# Ensure correct feature order expected by the model
X = X[model.feature_names_in_]

# Background sample for SHAP stability
background = X.sample(min(100, len(X)), random_state=42)

# Use LinearExplainer for linear models (Logistic Regression etc.)
explainer = shap.LinearExplainer(model, background)
shap_values = explainer.shap_values(X)

In [5]:
import matplotlib.pyplot as plt
import os

os.makedirs("../reports/SHAP", exist_ok=True)

plt.figure()
shap.summary_plot(shap_values, X, show=False)
plt.title("SHAP Global Feature Importance")

plt.savefig("../reports/SHAP/shap_summary.png", bbox_inches="tight")
plt.close()

In [6]:
plt.figure()
shap.summary_plot(shap_values, X, plot_type="bar", show=False)
plt.title("Top Features Driving Churn Prediction")

plt.savefig("../reports/SHAP/top_features.png", bbox_inches="tight")
plt.close()

In [7]:
churned_sample = X[y == 1].iloc[[0]]

plt.figure()
shap.force_plot(
    explainer.expected_value,
    explainer.shap_values(churned_sample),
    churned_sample,
    matplotlib=True,
    show=False
)

plt.savefig("../reports/SHAP/local_churned.png", bbox_inches="tight")
plt.close()

<Figure size 640x480 with 0 Axes>

In [8]:
retained_sample = X[y == 0].iloc[[0]]

plt.figure()
shap.force_plot(
    explainer.expected_value,
    explainer.shap_values(retained_sample),
    retained_sample,
    matplotlib=True,
    show=False
)

plt.savefig("../reports/SHAP/local_retained.png", bbox_inches="tight")
plt.close()

In [9]:
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt
import os

# Ensure SHAP directory exists
os.makedirs("../reports/SHAP", exist_ok=True)

# Top 3 features for PDP (rubric requirement)
top_features = ["tenure", "monthlycharges", "contract"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

PartialDependenceDisplay.from_estimator(
    model,
    X,
    features=top_features,
    grid_resolution=50,
    ax=axes
)

plt.suptitle("Partial Dependence Plots - Top 3 Features", fontsize=14, y=1.02)
plt.tight_layout()

# Save to uppercase SHAP path
plt.savefig("../reports/SHAP/pdp_top_features.png", bbox_inches="tight", dpi=300)
plt.close()

print("✓ PDP for top 3 features saved to ../reports/SHAP/pdp_top_features.png")

✓ PDP for top 3 features saved to ../reports/SHAP/pdp_top_features.png


## 💼 Business Questions & Insights  

### **1️⃣ Top 5 Churn Drivers**  
- **Contract type** → Month-to-month strongest  
- **Tenure** → Low tenure → high churn  
- **Monthly charges** → High cost sensitivity  
- **Tech support absence**  
- **Fiber optic service usage**  

---

### **2️⃣ High-Risk Segments**  
- **New customers (< 6 months tenure)**  
- **Month-to-month contracts**  
- **High monthly charges users**  
- **Fiber optic users without support**  

---

### **3️⃣ Pricing Strategy (from Regression Insight)**  
- **Reduce entry-level pricing for new users**  
- **Bundle services (internet + support discounts)**  
- **Incentivize long-term contracts**  

---

### **4️⃣ 🎯 Target 100 Customers Strategy**  

**Prioritize:**  
- High **SHAP churn probability**  
- High **revenue customers**  
- Low tenure but high charges  

**Risk Score:**  